# Vietnamese Clickbait Detector — Training Notebook
Fine-tune a small language model for binary clickbait classification on Google Colab.

## 1. Setup — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q torch transformers datasets peft bitsandbytes trl \
    accelerate scikit-learn pandas pyyaml pydantic tqdm matplotlib seaborn

print('Dependencies installed.')

In [ ]:
import os, sys

# Option A: clone from GitHub
# !git clone https://github.com/YOUR_USERNAME/clickbait-detector.git
# %cd clickbait-detector

# Option B: mount Google Drive where you have uploaded the project
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/clickbait-detector'  # <-- change this
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print('Working directory:', os.getcwd())

## 2. Upload Data & Configure

In [ ]:
# Upload articles.csv via Colab file browser, or copy from Drive:
import os
os.makedirs('data/raw', exist_ok=True)

# If uploading manually:
# from google.colab import files
# uploaded = files.upload()  # select articles.csv
# import shutil; shutil.move('articles.csv', 'data/raw/articles.csv')

# Or copy from Drive:
# !cp /content/drive/MyDrive/articles.csv data/raw/articles.csv

print('Data directory ready.')

In [ ]:
from src.config import get_config

# Load config with the Colab-optimized profile
config = get_config('configs/config.yaml', profile='colab_free')
print(f'Model: {config.model.name}')
print(f'Max length: {config.model.max_length}')
print(f'Batch size: {config.training.per_device_train_batch_size}')
print(f'Quantization: {config.quantization.bits}-bit')

## 3. Explore Data & Split

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(config.data.raw_csv_path)
print(f'Total samples: {len(df)}')
print('\nColumns:', df.columns.tolist())
print('\nSample rows:')
df[['title', 'lead_paragraph', 'label']].head(5)

In [ ]:
# Class distribution chart
counts = df['label'].value_counts()
fig, ax = plt.subplots(figsize=(5, 3))
counts.plot(kind='bar', ax=ax, color=['#2196F3', '#F44336'], edgecolor='black')
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
ax.set_xlabel('Label')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print(counts)

In [ ]:
from src.data_loader import load_raw_data, split_data, save_splits

df = load_raw_data(config.data.raw_csv_path)
ratios = {
    'train': config.data.split_ratios.train,
    'val': config.data.split_ratios.val,
    'test': config.data.split_ratios.test,
}
train_df, val_df, test_df = split_data(
    df, ratios=ratios,
    stratified=config.data.stratified,
    seed=config.data.seed,
)
save_splits(train_df, val_df, test_df, config.data.processed_dir)
print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

## 4. Train

In [ ]:
from src.utils import set_seed
from src.model import load_tokenizer, load_model, apply_lora
from src.data_loader import ClickbaitDataset

set_seed(config.training.seed)

tokenizer = load_tokenizer(config.model.name, config.model.trust_remote_code)
train_ds = ClickbaitDataset(train_df, tokenizer, config)
val_ds = ClickbaitDataset(val_df, tokenizer, config)
test_ds = ClickbaitDataset(test_df, tokenizer, config)

model = load_model(config)
model = apply_lora(model, config)
print('Model ready.')

In [ ]:
from src.trainer import build_trainer, run_training

trainer = build_trainer(model, tokenizer, train_ds, val_ds, config)
run_training(trainer, config)
print('Training complete!')

## 5. Evaluate

In [ ]:
from src.evaluate import full_evaluation
from src.model import load_finetuned_model
import os

best_ckpt = os.path.join(config.training.output_dir, 'checkpoints', 'best')
eval_model, eval_tokenizer = load_finetuned_model(config, checkpoint_path=best_ckpt)

metrics = full_evaluation(eval_model, eval_tokenizer, test_ds, config)
print('\nTest Metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
from IPython.display import Image
import os

cm_path = os.path.join(config.training.output_dir, 'results', 'confusion_matrix.png')
if os.path.exists(cm_path):
    display(Image(cm_path))

## 6. Save to Google Drive

In [ ]:
import shutil, os

drive_save_dir = '/content/drive/MyDrive/clickbait_checkpoints'
os.makedirs(drive_save_dir, exist_ok=True)

src_best = os.path.join(config.training.output_dir, 'checkpoints', 'best')
dst_best = os.path.join(drive_save_dir, 'best')

if os.path.exists(dst_best):
    shutil.rmtree(dst_best)
shutil.copytree(src_best, dst_best)
print(f'Checkpoint saved to {dst_best}')

# Also save results
src_results = os.path.join(config.training.output_dir, 'results')
dst_results = os.path.join(drive_save_dir, 'results')
if os.path.exists(dst_results):
    shutil.rmtree(dst_results)
shutil.copytree(src_results, dst_results)
print(f'Results saved to {dst_results}')